In [2]:
import pickle
import pandas as pd

In [3]:
df = pickle.load(open('dataset_lvl2.pkl','rb'))

In [4]:
df.columns

Index(['match_id', 'batting_team', 'ball', 'runs', 'player_dismissed', 'city',
       'venue', 'bowling_team', 'is_legal_ball'],
      dtype='object')

In [5]:
df.head()

,match_id,batting_team,ball,runs,player_dismissed,city,venue,bowling_team,is_legal_ball
0,2,Sunrisers Hyderabad,0.1,0,0,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,1
1,2,Sunrisers Hyderabad,0.2,0,0,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,1
2,2,Sunrisers Hyderabad,0.3,4,0,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,1
3,2,Sunrisers Hyderabad,0.4,0,0,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,1
4,2,Sunrisers Hyderabad,0.5,2,0,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,0


In [6]:
# batting team
# bowling team
# city
# curr score
# ball left
# run rate
# wicket left
# last 5 over runs

In [7]:
df.isnull().sum()

match_id               0
batting_team           0
ball                   0
runs                   0
player_dismissed       0
city                6343
venue                  0
bowling_team           0
is_legal_ball          0
dtype: int64

In [8]:
df[df['city'].isnull()]['venue'].value_counts()

venue
Dubai International Cricket Stadium    4106
Sharjah Cricket Stadium                2237
Name: count, dtype: int64

In [9]:
import numpy as np

df['city'] = np.where(
    df['city'].isnull(),
    df['venue'].str.split().str[0],
    df['city']
)

In [10]:
df['city'].isnull().sum()

np.int64(0)

In [11]:
df.drop(columns=['venue'],inplace=True)

In [12]:
df.head()

,match_id,batting_team,ball,runs,player_dismissed,city,bowling_team,is_legal_ball
0,2,Sunrisers Hyderabad,0.1,0,0,Hyderabad,Royal Challengers Bangalore,1
1,2,Sunrisers Hyderabad,0.2,0,0,Hyderabad,Royal Challengers Bangalore,1
2,2,Sunrisers Hyderabad,0.3,4,0,Hyderabad,Royal Challengers Bangalore,1
3,2,Sunrisers Hyderabad,0.4,0,0,Hyderabad,Royal Challengers Bangalore,1
4,2,Sunrisers Hyderabad,0.5,2,0,Hyderabad,Royal Challengers Bangalore,0


In [13]:
df['city'].value_counts()

city
Mumbai            23129
Delhi             12800
Kolkata           12797
Chennai           12191
Hyderabad         11134
Jaipur             8469
Bangalore          7869
Chandigarh         7326
Ahmedabad          6354
Pune               6050
Dubai              5729
Bengaluru          4609
Abu Dhabi          4441
Lucknow            3485
Sharjah            3482
Dharamsala         2313
Visakhapatnam      1991
Durban             1858
Centurion          1486
Rajkot             1229
Navi Mumbai        1133
Mohali             1090
Indore             1082
Raipur              987
New Chandigarh      967
Guwahati            943
Port Elizabeth      870
Johannesburg        870
Cuttack             856
Cape Town           813
Ranchi              716
Kochi               603
Kanpur              492
East London         380
Nagpur              370
Kimberley           368
Bloemfontein        251
Name: count, dtype: int64

In [14]:
cities = df['city'].value_counts()[df['city'].value_counts() > 600].index.tolist()

In [15]:
df = df[df['city'].isin(cities)]

In [16]:
df.head()

,match_id,batting_team,ball,runs,player_dismissed,city,bowling_team,is_legal_ball
0,2,Sunrisers Hyderabad,0.1,0,0,Hyderabad,Royal Challengers Bangalore,1
1,2,Sunrisers Hyderabad,0.2,0,0,Hyderabad,Royal Challengers Bangalore,1
2,2,Sunrisers Hyderabad,0.3,4,0,Hyderabad,Royal Challengers Bangalore,1
3,2,Sunrisers Hyderabad,0.4,0,0,Hyderabad,Royal Challengers Bangalore,1
4,2,Sunrisers Hyderabad,0.5,2,0,Hyderabad,Royal Challengers Bangalore,0


In [17]:
# now we will calculate curr_sum
df['runs'] = pd.to_numeric(df['runs'], errors='coerce')
df['curr_score'] = df.groupby('match_id')['runs'].cumsum()

In [18]:
df

,match_id,batting_team,ball,runs,player_dismissed,city,bowling_team,is_legal_ball,curr_score
0,2,Sunrisers Hyderabad,0.1,0,0,Hyderabad,Royal Challengers Bangalore,1,0
1,2,Sunrisers Hyderabad,0.2,0,0,Hyderabad,Royal Challengers Bangalore,1,0
2,2,Sunrisers Hyderabad,0.3,4,0,Hyderabad,Royal Challengers Bangalore,1,4
3,2,Sunrisers Hyderabad,0.4,0,0,Hyderabad,Royal Challengers Bangalore,1,4
4,2,Sunrisers Hyderabad,0.5,2,0,Hyderabad,Royal Challengers Bangalore,0,6
...,...,...,...,...,...,...,...,...,...
151528,1244,Sunrisers Hyderabad,19.2,6,0,Bangalore,Royal Challengers Bangalore,1,194
151529,1244,Sunrisers Hyderabad,19.3,6,0,Bangalore,Royal Challengers Bangalore,1,200
151530,1244,Sunrisers Hyderabad,19.4,1,0,Bangalore,Royal Challengers Bangalore,1,201
151531,1244,Sunrisers Hyderabad,19.5,1,0,Bangalore,Royal Challengers Bangalore,1,202


In [19]:
# balls left
# Calculate cumulative legal balls bowled
df['bowled_balls'] = (
    df.groupby('match_id')['is_legal_ball'].cumsum()
)

# Calculate balls remaining
df['ball_left'] = 120 - df['bowled_balls']

# Avoid negative values
df['ball_left'] = df['ball_left'].clip(lower=0)

In [60]:
# wickets left
df = df.copy()

# Calculate wickets lost per delivery

df['player_dismissed'] = (
    df['player_dismissed']
    .fillna('0')
    .astype(str)
    .ne('0')
    .astype(int)
)

# Calculate cumulative wickets lost in each match

df['player_dismissed'] = (
    df.groupby('match_id')['player_dismissed'].cumsum()
)

# Calculate wickets remaining

df['wickets_left'] = (
    10 - df['player_dismissed']
).clip(lower=0)

In [61]:
df.head()

,match_id,batting_team,ball,runs,player_dismissed,city,bowling_team,is_legal_ball,curr_score,bowled_balls,ball_left,wickets_left,curr,last_five
0,2,Sunrisers Hyderabad,0.1,0,0,Hyderabad,Royal Challengers Bangalore,1,0,1,119,10,0.0,0.0
1,2,Sunrisers Hyderabad,0.2,0,0,Hyderabad,Royal Challengers Bangalore,1,0,2,118,10,0.0,0.0
2,2,Sunrisers Hyderabad,0.3,4,0,Hyderabad,Royal Challengers Bangalore,1,4,3,117,10,8.0,4.0
3,2,Sunrisers Hyderabad,0.4,0,0,Hyderabad,Royal Challengers Bangalore,1,4,4,116,10,6.0,4.0
4,2,Sunrisers Hyderabad,0.5,2,0,Hyderabad,Royal Challengers Bangalore,0,6,4,116,10,9.0,6.0


In [42]:
# curr_run_rate'
# Calculate current run rate safely

df['curr'] = np.where(
    df['bowled_balls'] > 0,
    (df['curr_score'] * 6) / df['bowled_balls'],
    0
)

In [43]:
df

,match_id,batting_team,ball,runs,player_dismissed,city,bowling_team,is_legal_ball,curr_score,bowled_balls,ball_left,wickets_left,curr,last_five
0,2,Sunrisers Hyderabad,0.1,0,0,Hyderabad,Royal Challengers Bangalore,1,0,1,119,10,0.000000,0.0
1,2,Sunrisers Hyderabad,0.2,0,0,Hyderabad,Royal Challengers Bangalore,1,0,2,118,10,0.000000,0.0
2,2,Sunrisers Hyderabad,0.3,4,0,Hyderabad,Royal Challengers Bangalore,1,4,3,117,10,8.000000,4.0
3,2,Sunrisers Hyderabad,0.4,0,0,Hyderabad,Royal Challengers Bangalore,1,4,4,116,10,6.000000,4.0
4,2,Sunrisers Hyderabad,0.5,2,0,Hyderabad,Royal Challengers Bangalore,0,6,4,116,10,9.000000,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151528,1244,Sunrisers Hyderabad,19.2,6,81,Bangalore,Royal Challengers Bangalore,1,194,116,4,-71,10.034483,54.0
151529,1244,Sunrisers Hyderabad,19.3,6,82,Bangalore,Royal Challengers Bangalore,1,200,117,3,-72,10.256410,56.0
151530,1244,Sunrisers Hyderabad,19.4,1,83,Bangalore,Royal Challengers Bangalore,1,201,118,2,-73,10.220339,56.0
151531,1244,Sunrisers Hyderabad,19.5,1,84,Bangalore,Royal Challengers Bangalore,1,202,119,1,-74,10.184874,57.0


In [44]:
df['runs'].dtype


dtype('int64')

In [62]:
# now we want the score from last five overs
# Now we want the score from the last five overs
import numpy as np

# Calculate runs in the last 30 legal balls for each match

last_five = np.zeros(len(df))

for positions in df.groupby('match_id', sort=False).indices.values():

    runs = df.iloc[positions]['runs'].to_numpy(dtype=float)
    legal = df.iloc[positions]['is_legal_ball'].to_numpy(dtype=int)

    # Cumulative legal balls
    legal_cum = np.cumsum(legal)

    # Cumulative runs
    run_cum = np.concatenate(([0], np.cumsum(runs)))

    for i in range(len(positions)):

        cutoff = max(0, legal_cum[i] - 30)

        if cutoff == 0:
            start = 0
        else:
            start = np.searchsorted(
                legal_cum, cutoff, side='right'
            )

        last_five[positions[i]] = (
            run_cum[i + 1] - run_cum[start]
        )

df['last_five'] = last_five

In [63]:
df['last_five'].describe()

count    149672.000000
mean         36.141329
std          16.460060
min           0.000000
25%          26.000000
50%          37.000000
75%          47.000000
max         117.000000
Name: last_five, dtype: float64

In [64]:
df.head()

,match_id,batting_team,ball,runs,player_dismissed,city,bowling_team,is_legal_ball,curr_score,bowled_balls,ball_left,wickets_left,curr,last_five
0,2,Sunrisers Hyderabad,0.1,0,0,Hyderabad,Royal Challengers Bangalore,1,0,1,119,10,0.0,0.0
1,2,Sunrisers Hyderabad,0.2,0,0,Hyderabad,Royal Challengers Bangalore,1,0,2,118,10,0.0,0.0
2,2,Sunrisers Hyderabad,0.3,4,0,Hyderabad,Royal Challengers Bangalore,1,4,3,117,10,8.0,4.0
3,2,Sunrisers Hyderabad,0.4,0,0,Hyderabad,Royal Challengers Bangalore,1,4,4,116,10,6.0,4.0
4,2,Sunrisers Hyderabad,0.5,2,0,Hyderabad,Royal Challengers Bangalore,0,6,4,116,10,9.0,6.0


In [65]:
# remove unwanted columns
# df['batting_team','bowling_team','curr_score','ball_left','wickets_left','curr','last_five']]
# Calculate total runs scored in each match
total_score = (
    df.groupby('match_id')['runs']
    .sum()
    .reset_index()
)

# Merge total score with delivery-level data
final_df = total_score.merge(df, on='match_id')

# Select required columns
final_df = final_df[
    [
        'batting_team',
        'bowling_team',
        'curr_score',
        'ball_left',
        'wickets_left',
        'curr',
        'last_five',
        'city',
        'runs_x'
    ]
]

In [50]:
final_df = final_df[['batting_team','bowling_team','curr_score','ball_left','wickets_left','curr','last_five','city','runs_x']]

In [66]:
final_df

,batting_team,bowling_team,curr_score,ball_left,wickets_left,curr,last_five,city,runs_x
0,Sunrisers Hyderabad,Royal Challengers Bangalore,0,119,10,0.000000,0.0,Hyderabad,207
1,Sunrisers Hyderabad,Royal Challengers Bangalore,0,118,10,0.000000,0.0,Hyderabad,207
2,Sunrisers Hyderabad,Royal Challengers Bangalore,4,117,10,8.000000,4.0,Hyderabad,207
3,Sunrisers Hyderabad,Royal Challengers Bangalore,4,116,10,6.000000,4.0,Hyderabad,207
4,Sunrisers Hyderabad,Royal Challengers Bangalore,6,116,10,9.000000,6.0,Hyderabad,207
...,...,...,...,...,...,...,...,...,...
149667,Sunrisers Hyderabad,Royal Challengers Bangalore,194,4,0,10.034483,65.0,Bangalore,208
149668,Sunrisers Hyderabad,Royal Challengers Bangalore,200,3,0,10.256410,65.0,Bangalore,208
149669,Sunrisers Hyderabad,Royal Challengers Bangalore,201,2,0,10.220339,62.0,Bangalore,208
149670,Sunrisers Hyderabad,Royal Challengers Bangalore,202,1,0,10.184874,63.0,Bangalore,208


In [67]:
# resuflling the dataset
final_df  = final_df.sample(final_df.shape[0])

In [68]:
final_df.head()

,batting_team,bowling_team,curr_score,ball_left,wickets_left,curr,last_five,city,runs_x
148404,Kolkata Knight Riders,Royal Challengers Bangalore,91,54,0,8.272727,40.0,Kolkata,183
33817,Punjab Kings,Mumbai Indians,135,0,0,6.750000,30.0,Abu Dhabi,135
34645,Kolkata Knight Riders,Rajasthan Royals,105,44,0,8.289474,56.0,Sharjah,171
60437,Sunrisers Hyderabad,Rajasthan Royals,193,2,0,9.813559,62.0,Hyderabad,201
63498,Lucknow Super Giants,Delhi Capitals,151,44,0,11.921053,57.0,Visakhapatnam,209


In [69]:
# now we have to ml modelling
X = final_df.drop(columns=['runs_x'])
y = final_df['runs_x']
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [70]:
y_train

41846    152
16005    170
65563    221
90509    148
85992     67
        ... 
96349    171
48481    174
41438    158
78957    192
19426    155
Name: runs_x, Length: 119737, dtype: int64

In [71]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error


In [92]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
    --------------------------------------- 0.8/48.9 MB 1.5 MB/s eta 0:00:33
   - -------------------------------------- 1.3/48.9 MB 1.8 MB/s eta 0:00:27
   - -------------------------------------- 1.6/48.9 MB 1.9 MB/s eta 0:00:26
   - -------------------------------------- 2.1/48.9 MB 1.9 MB/s eta 0:00:25
   -- ------------------------------------- 2.6/48.9 MB 2.0 MB/s eta 0:00:24
   -- ------------------------------------- 3.1/48.9 MB 2.0 MB/s eta 0:00:23
   -- ------------------------------------- 3.7/48.9 MB 2.1 MB/s eta 0:00:22
   --- ------------------------------------ 3.9/48.9 MB 2.0 MB/s eta 0:00:23
   --- ------------------------------------ 4.2/48.9 MB 2.0 MB/s eta 0:00:23
   --- --------------------


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [72]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

trf = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first'),
     ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

In [73]:
# this is a pipiline through which our dataset will pass through at each step
pipe = Pipeline(steps=[
    ('step1',trf),  # heere  encoding will occur
    ('step2',StandardScaler()),  # here the scaling occur
    ('step3',XGBRegressor(n_estimators=1000,learning_rate=0.2,max_depth=12,random_state=1))
    #ghere the model get train
])

In [74]:

pipe.fit(x_train,y_train)
y_pred = pipe.predict(x_test)
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.919664740562439
4.443297386169434


In [75]:
# now exporting this model
pickle.dump(pipe,open('pipe.pkl','wb'))

In [76]:
import pickle

pipe = pickle.load(open('pipe.pkl', 'rb'))

In [77]:
import pandas as pd

input_data = pd.DataFrame({
    'batting_team': ['Mumbai Indians'],
    'bowling_team': ['Chennai Super Kings'],
    'curr_score': [120],
    'ball_left': [30],
    'wickets_left': [5],
    'curr': [8.0],
    'last_five': [45],
    'city': ['Mumbai']
})

In [78]:
predicted_score = pipe.predict(input_data)

print("Predicted First Innings Score:", predicted_score[0])

Predicted First Innings Score: 171.30281
